# Analysis 1.A — Treasury curve conditions and movements

Inspect actual 2Y, 10Y and 30Y yield history, choose exact dates, then compare net endpoint movements. Yields are percent; spread levels and all changes are basis points (1 percentage point = 100 bp).

**Start from the repository root:** run `poetry install`, then `poetry run jupyter lab`. Use this Poetry environment's Python kernel. The installed `bondview` package works from the notebook directory without path edits.

1. Set the inputs below. Choose `download` or `file`. The default cache is `data/fred/treasury_yields.json` under the repository root, excluded from Git. Custom paths should also stay outside Git.
2. Run imports and the data-loading cell once. `FRED_API_KEY` in the environment takes precedence over the root `.env`; without a key, the downloader explicitly uses FRED's public CSV endpoint. Downloading saves observations and metadata to one JSON file.
3. Run the chart and date-controls cells. Hover, zoom, or use the range slider to inspect the **historical lines**. Edit the two calendar controls to choose endpoints.
4. Manually run the final analysis cell after choosing dates. Changing dates and rerunning that cell uses memory and never downloads data.

`file` mode reads only the selected canonical JSON file, with no network request. Arbitrary CSV files are not inferred: the file must use the documented format in [README](README.md), including explicit Percent units and source/retrieval metadata. Empty observations stay missing. Errors in files are reported rather than corrected silently.

In [ ]:
from pathlib import Path
from datetime import date
from IPython.display import display, Markdown
import ipywidgets as widgets

from bondview.data.fred import download_fred, load_observations, save_observations
from bondview.analysis.treasury_curve import analyze_curve, display_table, interpret_curve
from bondview.plotting.treasury_curve import plot_yield_history

# Jupyter may start in the root or notebooks/ directory; package imports do not depend on this.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_MODE = "download"  # "download" or "file"
DATA_PATH = PROJECT_ROOT / "data/fred/treasury_yields.json"  # or an absolute local path
DOWNLOAD_START = "2024-01-01"
DOWNLOAD_END = "2025-01-31"
DISPLAY_START = "2024-09-01"
DISPLAY_END = "2025-01-31"
SLOPE_TOLERANCE_BP = 0.0

In [ ]:
# This is the only cell that can contact FRED.
if DATA_MODE == "download":
    observations = download_fred(DOWNLOAD_START, DOWNLOAD_END, env_path=PROJECT_ROOT / ".env")
    saved_path = save_observations(observations, DATA_PATH)
    print(f"Saved {len(observations)} dated rows to {saved_path}")
elif DATA_MODE == "file":
    observations = load_observations(DATA_PATH)
    print(f"Loaded {len(observations)} dated rows from {DATA_PATH}; no FRED request")
else:
    raise ValueError('DATA_MODE must be "download" or "file".')

metadata = observations.attrs["metadata"]
print(metadata["source"], "|", metadata.get("transport", "local file"))
print("Retrieved:", metadata["retrieved_at_utc"])
print("Loaded coverage:", observations.index.min().date(), "to", observations.index.max().date())
print("Missing observations per series:", observations.isna().sum().to_dict())
print("Retrieval-time snapshot; historical data availability at the observation date is not reconstructed.")

## Inspect history and choose endpoints

The initial chart view includes September 2024–January 2025. The loaded history remains available through zoom and the range slider. Lines keep missing values as gaps; no filling or interpolation is applied to observations. Select dates below, then execute the analysis cell manually. Dates outside loaded coverage, weekends, and holidays remain exact and can produce `N/A`.

In [ ]:
history_figure = plot_yield_history(observations, DISPLAY_START, DISPLAY_END)
history_figure.show()

In [ ]:
start_picker = widgets.DatePicker(description="Start", value=date(2024, 9, 16))
end_picker = widgets.DatePicker(description="End", value=date(2025, 1, 13))
display(widgets.HBox([start_picker, end_picker]))

## Analyze selected dates — run this cell manually

Each segment is classified independently using `Δslope = Δlong − Δshort`. The nonnegative `SLOPE_TOLERANCE_BP` applies only to slope changes (inclusive boundary); the default is zero. Yield direction uses actual signs. A stationary tenor is stated explicitly. “Opposing-direction twist” is a descriptive convention, not an exclusive universal taxonomy. Equal tenor changes are parallel shifts; approximately unchanged slopes with unequal changes keep their separate direction and slope descriptions.

Curve configuration describes each segment at each endpoint (positive = upward-sloping, negative = inverted, zero = flat). Available endpoint values survive even when their changes are unavailable. No observation is replaced with a nearby business day.

The design example uses Treasury par-yield observations for 2024-09-16 → 2025-01-13 and reports +84/+116/+103 bp and +32/−13 bp. This notebook calculates FRED/H.15 values as loaded; it never forces agreement. These snapshots do not reconstruct historical data vintages. This is Treasury-tenor market analysis, with no product-specific ETF return or recommendation.

In [ ]:
result = analyze_curve(
    observations, start_picker.value, end_picker.value,
    slope_tolerance_bp=SLOPE_TOLERANCE_BP,
)
display(display_table(result))
display(Markdown(interpret_curve(result)))
# Numeric start/end/change columns remain available in `result` for inspection.
# `result[["direction", "slope"]]` exposes directions separately from the movement label.